<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z328_FactorLatente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Factor Latente — Factor-Augmented HAR (FAVAR)

## La idea

Las 780 series de ventas no son independientes. Todas responden a **factores comunes no observables**:
- Un ciclo de demanda agregado del canal
- Estacionalidad del mercado
- Tendencia del negocio

Estos factores no se miden directamente, pero se pueden **extraer** de la covarianza entre series.

```
tn_{i,t} = λ_i1·F_1t + λ_i2·F_2t + ... + λ_iK·F_Kt + ε_{i,t}

F_kt  = factor latente k en período t  (común a todos los productos)
λ_ik  = carga del producto i en el factor k  (cuánto responde ese producto)
ε_{i,t} = ruido idiosincrático del producto i
```

## Pipeline

1. **Extraer factores** via PCA sobre la matriz (780 productos × T períodos)
2. **Interpretar** qué captura cada factor
3. **FAVAR**: augmentar las features HAR con los valores del factor en cada período
4. **Predicción**: proyectar los factores un paso adelante (AR simple), usarlos como features
5. **Submit** del modelo Factor-Augmented Panel

## ¿Por qué PCA y no Dynamic Factor Model (DFM)?

El DFM (Kalman filter, statsmodels) es más correcto teóricamente pero requiere supuestos sobre la dinámica del factor. Con solo ~36 períodos, PCA da factores igualmente interpretables y mucho más estable numéricamente. Si querés el DFM completo, ver Sección 8.

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle lightgbm

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
  import os
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  os.system(comando)

In [ ]:
import os
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

In [ ]:
PARAM = {
  'experimento':        'FactorLatente-01',
  'kaggle_competition': 'labo-iii-2026-rosario',
  'semilla_primigenia': 102191,
  # cuántos factores latentes extraer
  'n_factores': 5,
  # normalizar series antes de PCA? True = factores de correlacion, False = covarianza
  'normalizar': True,
  # modelo para el panel augmented: 'Ridge', 'LightGBM'
  'modelo': 'Ridge',
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2  Preparacion de datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")

tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])

periodos_sorted = tb_ventas["periodo"].unique().sort().to_list()
productos       = tb_apredecir["product_id"].to_list()

print(f"{tb_ventas.height} filas | {len(productos)} productos | {len(periodos_sorted)} períodos")
print(f"Períodos: {periodos_sorted[0]} → {periodos_sorted[-1]}")

# 3  Matriz productos × tiempo

Pivotamos: filas = productos, columnas = períodos.
Productos con historia incompleta (NaN) se rellenan con la mediana de ese producto.

Esta es la matriz de datos que PCA va a descomponer.

In [ ]:
# pivot: index=product_id, columns=periodo, values=tn
df_pivot = (
    tb_ventas.to_pandas()
    .pivot(index='product_id', columns='periodo', values='tn')
    .reindex(index=productos, columns=periodos_sorted)
)

# rellenar NaN con mediana por producto (eje=1)
df_pivot = df_pivot.apply(lambda row: row.fillna(row.median()), axis=1)

print(f"Matriz: {df_pivot.shape}  (productos × períodos)")
print(f"NaN restantes: {df_pivot.isna().sum().sum()}")
df_pivot.head(3)

# 4  Extracción de factores latentes via PCA

PCA sobre la matriz transpuesta (períodos × productos) extrae los **factores comunes en el tiempo**.

Con `normalizar=True` trabajamos sobre la matriz de correlación → cada producto tiene el mismo peso independientemente de su nivel de ventas. Con `normalizar=False` los productos con mayor volumen dominan.

**Interpretación econométrica:**
- Factor 1 = suele ser el nivel/tendencia común ("mercado total")
- Factor 2 = puede ser la diferencia entre categorías o ciclo estacional
- Factor 3+ = efectos más idiosincráticos

In [ ]:
mat = df_pivot.values  # shape: (n_productos, n_periodos)

if PARAM['normalizar']:
    # estandarizar cada producto (media 0, std 1) antes de PCA
    scaler_prod = StandardScaler()
    mat_sc = scaler_prod.fit_transform(mat.T).T  # (n_productos, n_periodos)
else:
    mat_sc = mat.copy()

# PCA sobre (períodos × productos) → factores son series de tiempo
pca = PCA(n_components=PARAM['n_factores'], random_state=PARAM['semilla_primigenia'])
factores = pca.fit_transform(mat_sc.T)  # shape: (n_periodos, n_factores)
cargas   = pca.components_.T            # shape: (n_productos, n_factores)

varianza_explicada = pca.explained_variance_ratio_
print("Varianza explicada por factor:")
for k, v in enumerate(varianza_explicada):
    print(f"  F{k+1}: {v*100:.1f}%  (acumulado: {varianza_explicada[:k+1].sum()*100:.1f}%)")

# DataFrames para manejo cómodo
df_factores = pd.DataFrame(
    factores,
    index=periodos_sorted,
    columns=[f'F{k+1}' for k in range(PARAM['n_factores'])]
)
df_cargas = pd.DataFrame(
    cargas,
    index=productos,
    columns=[f'F{k+1}' for k in range(PARAM['n_factores'])]
)

display(df_factores.tail(5))

## 4.1  Visualización de factores en el tiempo

Cada factor es una serie temporal — el "estado del mercado" en cada período.
Si son interpretables esperamos:
- F1: tendencia suave (mercado total)
- F2: estacionalidad (picos anuales)
- F3+: shocks o efectos de grupos de productos

In [ ]:
fig, axes = plt.subplots(PARAM['n_factores'], 1,
                          figsize=(14, 2.5 * PARAM['n_factores']),
                          sharex=True)

colores = cm.tab10.colors
x = range(len(periodos_sorted))
xtick_labels = [str(p) for p in periodos_sorted]

for k in range(PARAM['n_factores']):
    ax = axes[k] if PARAM['n_factores'] > 1 else axes
    ax.plot(x, df_factores[f'F{k+1}'].values, color=colores[k], linewidth=1.8)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_ylabel(f'F{k+1}', fontsize=9)
    ax.set_title(
        f'Factor {k+1}  ({varianza_explicada[k]*100:.1f}% varianza)',
        fontsize=9
    )
    ax.set_xticks(x[::3])
    ax.set_xticklabels(xtick_labels[::3], rotation=45, ha='right', fontsize=7)

fig.suptitle('Factores latentes en el tiempo (PCA sobre ventas de productos)', fontsize=11)
plt.tight_layout()
plt.show()

## 4.2  Cargas: ¿qué productos definen cada factor?

Los productos con carga alta en F1 son los que más determinan el factor común.
Los con carga positiva y negativa en F2 son los que se "compensan" entre sí.

In [ ]:
fig, axes = plt.subplots(1, min(PARAM['n_factores'], 3), figsize=(14, 4))
if PARAM['n_factores'] == 1:
    axes = [axes]

for k in range(min(PARAM['n_factores'], 3)):
    cargas_k = df_cargas[f'F{k+1}'].sort_values()
    n_show = 15
    top    = cargas_k.nlargest(n_show)
    bottom = cargas_k.nsmallest(n_show)
    mostrar = pd.concat([bottom, top])

    colores_bar = ['tomato' if v < 0 else 'steelblue' for v in mostrar.values]
    axes[k].barh(range(len(mostrar)), mostrar.values, color=colores_bar)
    axes[k].set_yticks(range(len(mostrar)))
    axes[k].set_yticklabels([str(p) for p in mostrar.index], fontsize=6)
    axes[k].axvline(0, color='black', linewidth=0.5)
    axes[k].set_title(f'F{k+1}: cargas extremas (top/bottom {n_show})', fontsize=9)

plt.suptitle('Productos que definen cada factor', fontsize=11)
plt.tight_layout()
plt.show()

# 5  Proyección de factores hacia el futuro

Para usar los factores como features en la predicción de 202001 y 202002
necesitamos su valor en esos períodos — pero no existen todavía.

Estrategia: ajustamos un **AR(1) simple** a cada factor y proyectamos 2 pasos.

Alternativa más simple: usar el último valor conocido (random walk).

In [ ]:
from statsmodels.tsa.ar_model import AutoReg

factores_futuros = {}  # factor_k -> [pred_t+1, pred_t+2]

for k in range(PARAM['n_factores']):
    serie_f = df_factores[f'F{k+1}'].values
    try:
        ar = AutoReg(serie_f, lags=1, old_names=False).fit()
        preds = ar.forecast(2)
        factores_futuros[f'F{k+1}'] = [float(preds[0]), float(preds[1])]
    except Exception:
        # fallback: random walk
        factores_futuros[f'F{k+1}'] = [float(serie_f[-1]), float(serie_f[-1])]

print("Proyección de factores:")
print(f"{'Factor':>8}  {'202001':>10}  {'202002':>10}  {'201912 (real)':>14}")
for k in range(PARAM['n_factores']):
    fname = f'F{k+1}'
    real_last = df_factores[fname].iloc[-1]
    p1, p2 = factores_futuros[fname]
    print(f"{fname:>8}  {p1:>10.3f}  {p2:>10.3f}  {real_last:>14.3f}")

# 6  FAVAR: panel con factores como features adicionales

Extendemos el panel de z323 agregando el valor del factor en cada período `t`.

```
features HAR:    lag1, lag2, lag3, mean_3m, mean_6m, mean_12m, mes, product_id
features nuevas: F1_t, F2_t, ..., FK_t   ← estado del mercado en t
```

El modelo aprende cuánto importa el estado del mercado para predecir cada producto.

In [ ]:
# diccionario periodo -> valores de factores
factor_cols = [f'F{k+1}' for k in range(PARAM['n_factores'])]
factor_by_periodo = {p: df_factores.loc[p, factor_cols].to_dict()
                     for p in periodos_sorted}

def safe_mean(arr):
    return float(arr.mean()) if len(arr) > 0 else 0.0

filas = []
for pid in productos:
    serie_df = tb_ventas.filter(pl.col("product_id") == pid).sort("periodo")
    periodos_prod = serie_df["periodo"].to_list()
    tn = serie_df["tn"].to_numpy().astype(float)
    T  = len(tn)

    for t in range(12, T - 2):
        periodo_t      = periodos_prod[t]
        periodo_target = periodos_prod[t + 2]
        mes_target = int(str(periodo_target)[4:6])

        fila = {
            'product_id':  pid,
            'periodo_t':   periodo_t,
            'lag1':        tn[t - 1],
            'lag2':        tn[t - 2],
            'lag3':        tn[t - 3],
            'mean_3m':     safe_mean(tn[t-3:t]),
            'mean_6m':     safe_mean(tn[t-6:t]),
            'mean_12m':    safe_mean(tn[t-12:t]),
            'mes':         mes_target,
            'tn_t2':       tn[t + 2]
        }
        # agregar factores del período t
        if periodo_t in factor_by_periodo:
            fila.update(factor_by_periodo[periodo_t])
        else:
            for fc in factor_cols:
                fila[fc] = 0.0

        filas.append(fila)

tb_panel = pl.DataFrame(filas)
print(f"Panel FAVAR: {tb_panel.height} filas x {tb_panel.width} columnas")
display(tb_panel.head(3))

# 7  Entrenamiento y validación

In [ ]:
periodo_corte = tb_panel["periodo_t"].sort(descending=True).unique()[6]

tb_train = tb_panel.filter(pl.col("periodo_t") <= periodo_corte)
tb_val   = tb_panel.filter(pl.col("periodo_t") >  periodo_corte)

FEATURES_HAR    = ['lag1', 'lag2', 'lag3', 'mean_3m', 'mean_6m', 'mean_12m', 'mes', 'product_id']
FEATURES_FAVAR  = FEATURES_HAR + factor_cols
TARGET = 'tn_t2'

X_train_har   = tb_train.select(FEATURES_HAR).to_numpy()
X_train_favar = tb_train.select(FEATURES_FAVAR).to_numpy()
y_train       = tb_train[TARGET].to_numpy()

X_val_har   = tb_val.select(FEATURES_HAR).to_numpy()
X_val_favar = tb_val.select(FEATURES_FAVAR).to_numpy()
y_val       = tb_val[TARGET].to_numpy()

print(f"Train: {tb_train.height}  |  Val: {tb_val.height}")
print(f"Features HAR: {len(FEATURES_HAR)}  |  Features FAVAR: {len(FEATURES_FAVAR)}")

In [ ]:
if PARAM['modelo'] == 'Ridge':
    scaler_har   = StandardScaler()
    scaler_favar = StandardScaler()

    X_train_har_sc   = scaler_har.fit_transform(X_train_har)
    X_train_favar_sc = scaler_favar.fit_transform(X_train_favar)

    alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

    m_har   = RidgeCV(alphas=alphas, cv=5).fit(X_train_har_sc, y_train)
    m_favar = RidgeCV(alphas=alphas, cv=5).fit(X_train_favar_sc, y_train)

    pred_val_har   = m_har.predict(scaler_har.transform(X_val_har))
    pred_val_favar = m_favar.predict(scaler_favar.transform(X_val_favar))

elif PARAM['modelo'] == 'LightGBM':
    m_har = lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=31,
        random_state=PARAM['semilla_primigenia'], verbose=-1
    ).fit(X_train_har, y_train,
          eval_set=[(X_val_har, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False)])

    m_favar = lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=31,
        random_state=PARAM['semilla_primigenia'], verbose=-1
    ).fit(X_train_favar, y_train,
          eval_set=[(X_val_favar, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False)])

    pred_val_har   = m_har.predict(X_val_har)
    pred_val_favar = m_favar.predict(X_val_favar)

rmse_har   = np.sqrt(mean_squared_error(y_val, pred_val_har))
rmse_favar = np.sqrt(mean_squared_error(y_val, pred_val_favar))

print(f"RMSE validación HAR solo : {rmse_har:.4f}")
print(f"RMSE validación FAVAR    : {rmse_favar:.4f}")
print(f"Mejora: {(rmse_har - rmse_favar) / rmse_har * 100:+.1f}%")

## 7.1  Importancia de los factores

¿Cuánto aportan los factores latentes sobre las features HAR?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if PARAM['modelo'] == 'Ridge':
    # coeficientes estandarizados
    coefs = m_favar.coef_
    colores_bar = ['tomato' if f.startswith('F') else 'steelblue' for f in FEATURES_FAVAR]
    axes[0].barh(FEATURES_FAVAR, coefs, color=colores_bar)
    axes[0].axvline(0, color='black', linewidth=0.5)
    axes[0].set_title('Coeficientes Ridge FAVAR (rojo = factores latentes)')

elif PARAM['modelo'] == 'LightGBM':
    imp = m_favar.feature_importances_
    colores_bar = ['tomato' if f.startswith('F') else 'steelblue' for f in FEATURES_FAVAR]
    axes[0].barh(FEATURES_FAVAR, imp, color=colores_bar)
    axes[0].set_title('Importancia LightGBM FAVAR (rojo = factores latentes)')

# comparacion RMSE
axes[1].bar(['HAR solo', 'FAVAR (HAR + factores)'],
            [rmse_har, rmse_favar],
            color=['steelblue', 'darkorange'])
axes[1].set_ylabel('RMSE validación')
axes[1].set_title('Impacto de los factores latentes en RMSE')
for i, v in enumerate([rmse_har, rmse_favar]):
    axes[1].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

# 8  Predicción para 202002

Usamos los factores proyectados a t+2 (202002) calculados en la sección 5.

In [ ]:
# valores de factores para 202002 (t+2 desde 201912)
factores_202002 = {fc: factores_futuros[fc][1] for fc in factor_cols}

resultados = []

for pid in productos:
    serie = (
        tb_ventas.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    t = len(serie) - 1

    fila_har = [
        serie[t - 1] if t >= 1 else 0.0,
        serie[t - 2] if t >= 2 else 0.0,
        serie[t - 3] if t >= 3 else 0.0,
        safe_mean(serie[max(0, t-3):t]),
        safe_mean(serie[max(0, t-6):t]),
        safe_mean(serie[max(0, t-12):t]),
        2,   # mes = febrero
        pid
    ]
    fila_favar = fila_har + [factores_202002[fc] for fc in factor_cols]

    X_pred = np.array([fila_favar])

    if PARAM['modelo'] == 'Ridge':
        pred = float(m_favar.predict(scaler_favar.transform(X_pred))[0])
    else:
        pred = float(m_favar.predict(X_pred)[0])

    resultados.append({'product_id': pid, 'tn': max(pred, 0.0)})

tb_final = pl.DataFrame(resultados)
display(tb_final.head(10))
print(f"Nulls: {tb_final['tn'].is_null().sum()}")

# 9  Submit a Kaggle

In [ ]:
archivo = f"FAVAR_{PARAM['modelo']}_K{PARAM['n_factores']}.csv"
mensaje = f"Factor-Augmented HAR K={PARAM['n_factores']} factores PCA + {PARAM['modelo']}"

tb_final.write_csv(archivo)
kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)

# 10  [Opcional] Dynamic Factor Model con Kalman Filter

Si querés el DFM completo (más riguroso teóricamente), `statsmodels` lo tiene.
Requiere más tiempo de ajuste y convergencia más delicada con pocos períodos.

In [ ]:
# CELDA OPCIONAL — correr solo si querés comparar con DFM Kalman

# !uv pip install -q statsmodels
# from statsmodels.tsa.statespace.dynamic_factor_mq import DynamicFactorMQ

# df_for_dfm = df_pivot.T  # (períodos × productos)

# # DFM con K factores, factor_order=1 (AR(1) en el factor)
# dfm = DynamicFactorMQ(df_for_dfm,
#                       factors=PARAM['n_factores'],
#                       factor_order=1)
# res_dfm = dfm.fit(disp=False, maxiter=200)

# # factores suavizados
# smoothed = res_dfm.factors['smoothed']
# print(smoothed.tail())

# # forecast 2 pasos
# fc_dfm = res_dfm.forecast(2)
# print(fc_dfm)

print("DFM opcional — descomentar para usar")

# 11  Qué probar si el score no mejora

| Cambio | Donde | Por qué |
|---|---|---|
| `n_factores: 3` | PARAM | Menos factores → menos ruido en la proyección |
| `n_factores: 10` | PARAM | Más factores → captura grupos de productos más finos |
| `normalizar: False` | PARAM | PCA de covarianza → productos con mayor volumen dominan el factor |
| `'modelo': 'LightGBM'` | PARAM | Captura interacciones factor × producto |
| Agregar factor a AutoGluon como known_covariate | z324 | Los modelos profundos (TFT, DeepAR) pueden explotar mejor los factores |
| Proyectar factores con ARIMA en vez de AR(1) | sección 5 | Mejor proyección del estado de mercado futuro |
| PCA sobre log1p(tn) | sección 3 | Factores en escala log → menos dominados por productos de alto volumen |
| Agregar factor como static feature (carga λ_i) | — | La carga de un producto en el factor es su "sensibilidad al mercado" |